# 49 — Embedding Evaluation & Benchmark
**Goal:** Build a systematic benchmark comparing embedding models on resume matching tasks.

Chapters 46–48 used `all-MiniLM-L6-v2` as the embedding model, but "it works" is not a decision criterion. Different models trade accuracy, speed, and size differently, and a model that ranks well on generic text may rank poorly on resume–JD pairs. This chapter builds a tiny, repeatable benchmark: fixed resume–JD pairs with human-judged match scores, run through any model, and summarized with one number — RMSE.

**Why it matters for resumes / ATS:** the embedding model is the foundation of every score in this block; a weak foundation caps the quality of matching, retrieval, and filtering no matter how clever the surrounding code. Benchmarking makes the choice evidence-based and, repeated over time, catches regressions when models or libraries are upgraded — a model that silently degrades is worse than one you never trusted.

## 1. Defining the Benchmark

A benchmark is only as good as its ground truth. Here, each pair is `(resume_text, jd_text, expected_score)` where `expected` is a human judgment on a 0–1 scale — how well the resume matches the JD.

**What the code does:** defines `eval_pairs` with five cases deliberately spanning the difficulty range:
- two strong matches (`0.9`, `0.8`) — Python/NLP and Java/Spring,
- one partial match (`0.6`) — DevOps vs cloud infrastructure,
- two clear mismatches (`0.2`, `0.1`) — data scientist vs frontend, project manager vs ML engineer.

**Try it:** these pairs are the test set of the chapter. Notice they stress *semantic* understanding, not keyword overlap — "Data scientist with TensorFlow" and "Frontend React developer" share almost no vocabulary, so a bag-of-words model fails where a good embedding model succeeds.

In [ ]:
# Evaluation pairs: (resume_text, jd_text, expected_match_score 0-1)
eval_pairs = [
    ("Python developer with NLP experience", "Looking for Python NLP engineer", 0.9),
    ("Java backend developer with Spring", "Senior Java developer Spring Boot", 0.8),
    ("Data scientist with TensorFlow", "Frontend React developer", 0.2),
    ("DevOps engineer Docker Kubernetes AWS", "Cloud infrastructure engineer", 0.6),
    ("Project manager with agile expertise", "Python ML engineer", 0.1),
]
print(f"Benchmark has {len(eval_pairs)} pairs")

## 2. Evaluating Models

`evaluate_model()` is the harness: load a model, encode both sides of every pair, compare the cosine similarity to the human label, and reduce all errors to a single RMSE.

**What the code does:**
- For each pair: `model.encode(resume)` and `model.encode(jd)`, then `util.cos_sim(emb1, emb2).item()` — the model's predicted match score.
- Records `abs(sim - expected)` per pair and computes `RMSE = sqrt(mean((sim - expected)^2))` — lower is better, and squaring punishes large disagreements disproportionately.
- Wraps everything in `try/except`; a model that fails to load (e.g. no network for the HuggingFace download) returns `RMSE = inf` and is reported as SKIPPED rather than crashing the loop.

**Expected:** three models are attempted — `all-MiniLM-L6-v2`, `all-mpnet-base-v2`, `multi-qa-MiniLM-L6-cos-v1`. The first run downloads each from HuggingFace, so network access is required; on a machine without it you will see the SKIPPED path, which is the harness working as designed.

In [ ]:
from sentence_transformers import SentenceTransformer, util
import numpy as np

def evaluate_model(model_name):
    try:
        model = SentenceTransformer(model_name)
        scores = []
        for resume, jd, expected in eval_pairs:
            emb1 = model.encode(resume)
            emb2 = model.encode(jd)
            sim = util.cos_sim(emb1, emb2).item()
            scores.append((resume, jd, sim, expected, abs(sim - expected)))
        rmse = np.sqrt(np.mean([(s - e)**2 for _, _, s, e, _ in scores]))
        return scores, rmse
    except Exception as e:
        return None, float('inf')

print("Evaluating models...")
for model_name in ["all-MiniLM-L6-v2", "all-mpnet-base-v2", "multi-qa-MiniLM-L6-cos-v1"]:
    scores, rmse = evaluate_model(model_name)
    if scores:
        print(f"\n{model_name:35s} RMSE: {rmse:.3f}")
        for r, j, s, e, _ in scores:
            print(f"  {r[:30]:30s} vs {j[:30]:30s} -> {s:.2f} (expected {e})")
    else:
        print(f"\n{model_name:35s} SKIPPED (not available)")

## 3. Results Visualization

Numbers alone are hard to compare; the chapter closes by rendering RMSE as a sorted, ASCII bar chart.

**What the code does:**
- Uses a hardcoded `results` dict (`all-MiniLM-L6-v2: 0.12`, `all-mpnet-base-v2: 0.09`, `multi-qa-MiniLM-L6-cos-v1: 0.15`) — **note these are illustrative placeholders baked into the notebook, not the output of Section 2**; in a real run you would fill this dict from `evaluate_model()` results.
- Sorts by RMSE ascending (best first) and draws `int((1 - rmse) * 20)` bar characters per model.
- Ends with the code's own takeaway: `all-mpnet-base-v2` is often the most accurate but slower, while `all-MiniLM-L6-v2` is the production speed/accuracy trade-off.

**Try it:** replace the hardcoded dict with the real scores from Section 2 and the chart updates automatically — that is the whole point of keeping the visualization separate from the evaluation.

In [ ]:
# Results summary
results = {"all-MiniLM-L6-v2": 0.12, "all-mpnet-base-v2": 0.09, "multi-qa-MiniLM-L6-cos-v1": 0.15}
print("\nModel comparison (lower RMSE = better):")
for model, rmse in sorted(results.items(), key=lambda x: x[1]):
    bar = "|" * int((1 - rmse) * 20)
    print(f"  {model:35s} RMSE={rmse:.2f}  {bar}")

print("\nKey insight: all-mpnet-base-v2 often best but slower.")
print("all-MiniLM-L6-v2 is the best speed/accuracy tradeoff for production.")

## Summary: Systematic benchmarks prevent regression. Track RMSE across model versions.

This chapter made model choice a measurement instead of an opinion: a fixed set of human-labeled resume–JD pairs, a reusable `evaluate_model()` harness, and RMSE as the single comparison metric. The same harness rerun after a model, library, or data change tells you immediately whether the system got better or worse. That discipline is what separates a maintained ATS from a demo — the benchmark is the regression test for the entire embedding layer built in Ch. 46–48.

## Key Insight

**Benchmark before you trust — an embedding model is a scored hypothesis, not a fact.**

RMSE over a small human-labeled pair set is a proxy for matching quality, and tracking it across model versions turns "which model?" into a number, not a preference. The practical winner is usually a speed/accuracy compromise: `all-MiniLM-L6-v2` for production latency, heavier models like `all-mpnet-base-v2` when quality dominates. With retrieval (47), storage (48), and evaluation (49) in place, Ch. 50 — ATS Rule Design — assembles them into the screening rules a hiring system enforces.